# LSTM PyTorch 실습

이 노트북은 LSTM의 핵심인 기억을 작은 예제로 직접 확인하는 실습입니다.

학습 목표
- LSTM이 시퀀스 정보를 어떻게 받는지 이해하기
- `nn.LSTM`으로 간단한 모델 만들기
- 긴 문맥에서 첫 정보를 기억하는지 확인하기
- 필요하면 `nn.RNN`과 비교해 보기

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cpu


## 1) 실습 문제

입력은 무작위 숫자 시퀀스입니다. 정답은 맨 처음 숫자입니다.

예를 들어 `[3, 8, 1, 4, 7]` 이 들어오면 정답은 `3`입니다.
이 문제는 뒤쪽 숫자들이 많아질수록 앞의 정보를 잘 기억해야 풀 수 있습니다.

In [7]:
def generate_memory_dataset(num_samples=4000, seq_len=12, vocab_size=5):
    x = torch.randint(0, vocab_size, (num_samples, seq_len), dtype=torch.long)
    y = x[:, 0].clone()
    return x, y

seq_len = 12
vocab_size = 5
train_x, train_y = generate_memory_dataset(num_samples=4000, seq_len=seq_len, vocab_size=vocab_size)
val_x, val_y = generate_memory_dataset(num_samples=1000, seq_len=seq_len, vocab_size=vocab_size)

train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=128, shuffle=True)

print(train_x.shape, train_y.shape)
print("예시 입력:", train_x[0].tolist())
print("정답:", train_y[0].item())

torch.Size([4000, 12]) torch.Size([4000])
예시 입력: [3, 1, 2, 0, 0, 2, 2, 2, 0, 0, 1, 0]
정답: 3


## 2) LSTM 모델 만들기

구조는 다음과 같습니다.

`Embedding -> LSTM -> Linear -> 분류`

마지막 시점의 출력을 사용해 처음 숫자를 맞히는 구조입니다.

In [8]:
class SequenceMemoryModel(nn.Module):
    def __init__(self, model_type="LSTM", vocab_size=10, emb_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.model_type = model_type

        if model_type == "LSTM":
            self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        elif model_type == "RNN":
            self.rnn = nn.RNN(emb_dim, hidden_dim, batch_first=True)
        else:
            raise ValueError(f"Unknown model_type: {model_type}")

        self.classifier = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, hidden = self.rnn(embedded)
        last_output = outputs[:, -1, :]
        logits = self.classifier(last_output)
        return logits, hidden

demo_model = SequenceMemoryModel(model_type="LSTM", vocab_size=vocab_size).to(DEVICE)
print(demo_model)

SequenceMemoryModel(
  (embedding): Embedding(5, 32)
  (rnn): LSTM(32, 64, batch_first=True)
  (classifier): Linear(in_features=64, out_features=5, bias=True)
)


## 3) LSTM 내부 출력 모양 확인

`nn.LSTM`은 마지막 출력뿐 아니라 hidden state와 cell state도 돌려줍니다.
이 셀은 실제 텐서 모양을 확인해서 개념과 연결해 보는 용도입니다.

In [9]:
sample_x = train_x[:2].to(DEVICE)
sample_embedding = demo_model.embedding(sample_x)
sample_outputs, (sample_h, sample_c) = demo_model.rnn(sample_embedding)

print("input shape:", sample_x.shape)
print("embedding shape:", sample_embedding.shape)
print("outputs shape:", sample_outputs.shape)
print("hidden shape:", sample_h.shape)
print("cell shape:", sample_c.shape)

input shape: torch.Size([2, 12])
embedding shape: torch.Size([2, 12, 32])
outputs shape: torch.Size([2, 12, 64])
hidden shape: torch.Size([1, 2, 64])
cell shape: torch.Size([1, 2, 64])


## 4) 학습 함수

아래 함수는 LSTM과 RNN을 같은 조건으로 학습시킵니다.
정확도도 함께 출력해서 두 모델을 비교할 수 있게 했습니다.

In [10]:
def accuracy(logits, target):
    pred = logits.argmax(dim=-1)
    return (pred == target).float().mean().item()

@torch.no_grad()
def evaluate(model, x, y, criterion):
    model.eval()
    x = x.to(DEVICE)
    y = y.to(DEVICE)
    logits, _ = model(x)
    loss = criterion(logits, y)
    acc = accuracy(logits, y)
    return loss.item(), acc

def train_model(model_type="LSTM", epochs=15, lr=0.01, hidden_dim=96, emb_dim=32):
    model = SequenceMemoryModel(model_type=model_type, vocab_size=vocab_size, emb_dim=emb_dim, hidden_dim=hidden_dim).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_acc = 0.0
        num_batches = 0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()
            logits, _ = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()
            running_acc += accuracy(logits, batch_y)
            num_batches += 1

        train_loss = running_loss / num_batches
        train_acc = running_acc / num_batches
        val_loss, val_acc = evaluate(model, val_x, val_y, criterion)
        history.append((train_loss, train_acc, val_loss, val_acc))

        print(f"[{model_type}] epoch {epoch:02d}/{epochs} | train loss {train_loss:.4f} acc {train_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")

    return model, history

## 5) LSTM 학습

이제 실제로 학습시켜 봅니다.
보통 이 과제에서는 LSTM이 RNN보다 긴 의존성을 더 안정적으로 기억합니다.

In [11]:
lstm_model, lstm_history = train_model(model_type="LSTM", epochs=15, lr=0.01, hidden_dim=96, emb_dim=32)

[LSTM] epoch 01/15 | train loss 1.6154 acc 0.2009 | val loss 1.6070 acc 0.2460
[LSTM] epoch 02/15 | train loss 1.3264 acc 0.3540 | val loss 1.1089 acc 0.4190
[LSTM] epoch 03/15 | train loss 1.1197 acc 0.4048 | val loss 1.1004 acc 0.4160
[LSTM] epoch 04/15 | train loss 1.1129 acc 0.4087 | val loss 1.0761 acc 0.4420
[LSTM] epoch 05/15 | train loss 1.0021 acc 0.6001 | val loss 0.7663 acc 0.6960
[LSTM] epoch 06/15 | train loss 0.4050 acc 0.7874 | val loss 0.2579 acc 0.9060
[LSTM] epoch 07/15 | train loss 0.1090 acc 0.9683 | val loss 0.0023 acc 1.0000
[LSTM] epoch 08/15 | train loss 0.0149 acc 0.9968 | val loss 0.0016 acc 1.0000
[LSTM] epoch 09/15 | train loss 0.0009 acc 1.0000 | val loss 0.0005 acc 1.0000
[LSTM] epoch 10/15 | train loss 0.0004 acc 1.0000 | val loss 0.0003 acc 1.0000
[LSTM] epoch 11/15 | train loss 0.0003 acc 1.0000 | val loss 0.0003 acc 1.0000
[LSTM] epoch 12/15 | train loss 0.0002 acc 1.0000 | val loss 0.0002 acc 1.0000
[LSTM] epoch 13/15 | train loss 0.0002 acc 1.0000 | 

## 6) 결과 확인

몇 개 샘플을 뽑아서 첫 숫자를 맞히는지 확인합니다.

In [13]:
with torch.no_grad():
    lstm_model.eval()
    sample_inputs = val_x[:8].to(DEVICE)
    sample_targets = val_y[:8].to(DEVICE)
    sample_logits, _ = lstm_model(sample_inputs)
    sample_predictions = sample_logits.argmax(dim=-1)

    for index in range(8):
        print(f"input={sample_inputs[index].tolist()} | target={sample_targets[index].item()} | pred={sample_predictions[index].item()}")

input=[2, 1, 2, 3, 3, 4, 0, 1, 3, 2, 3, 1] | target=2 | pred=2
input=[0, 3, 3, 0, 2, 3, 2, 3, 1, 3, 1, 4] | target=0 | pred=0
input=[1, 2, 0, 3, 2, 0, 4, 2, 4, 4, 1, 2] | target=1 | pred=1
input=[2, 1, 3, 3, 2, 0, 4, 0, 0, 2, 0, 3] | target=2 | pred=2
input=[4, 2, 2, 1, 2, 0, 3, 2, 2, 4, 1, 2] | target=4 | pred=4
input=[4, 0, 1, 2, 0, 0, 3, 3, 2, 4, 1, 1] | target=4 | pred=4
input=[1, 3, 3, 1, 0, 1, 1, 2, 4, 3, 2, 0] | target=1 | pred=1
input=[4, 4, 0, 4, 3, 0, 0, 4, 1, 0, 4, 0] | target=4 | pred=4


## 7) RNN과 비교해 보기

같은 문제를 `nn.RNN`으로도 학습해 보면 차이를 체감하기 쉽습니다.
실행 시간이 부담되면 이 셀은 건너뛰어도 됩니다.

In [14]:
rnn_model, rnn_history = train_model(model_type="RNN", epochs=15, lr=0.01, hidden_dim=96, emb_dim=32)

[RNN] epoch 01/15 | train loss 1.2323 acc 0.4353 | val loss 0.0363 acc 0.9970
[RNN] epoch 02/15 | train loss 0.1638 acc 0.9709 | val loss 0.1384 acc 0.8990
[RNN] epoch 03/15 | train loss 0.1986 acc 0.9312 | val loss 0.1174 acc 0.9190
[RNN] epoch 04/15 | train loss 0.0392 acc 0.9739 | val loss 0.0013 acc 1.0000
[RNN] epoch 05/15 | train loss 0.0008 acc 1.0000 | val loss 0.0004 acc 1.0000
[RNN] epoch 06/15 | train loss 0.0003 acc 1.0000 | val loss 0.0003 acc 1.0000
[RNN] epoch 07/15 | train loss 0.0003 acc 1.0000 | val loss 0.0002 acc 1.0000
[RNN] epoch 08/15 | train loss 0.0002 acc 1.0000 | val loss 0.0002 acc 1.0000
[RNN] epoch 09/15 | train loss 0.0002 acc 1.0000 | val loss 0.0002 acc 1.0000
[RNN] epoch 10/15 | train loss 0.0002 acc 1.0000 | val loss 0.0002 acc 1.0000
[RNN] epoch 11/15 | train loss 0.0001 acc 1.0000 | val loss 0.0001 acc 1.0000
[RNN] epoch 12/15 | train loss 0.0001 acc 1.0000 | val loss 0.0001 acc 1.0000
[RNN] epoch 13/15 | train loss 0.0001 acc 1.0000 | val loss 0.00

## 8) 마무리

이 실습에서 확인한 점
- LSTM은 긴 시퀀스에서도 정보를 더 잘 유지합니다.
- `nn.LSTM`은 `output`, `hidden state`, `cell state`를 함께 돌려줍니다.
- 같은 구조라도 RNN보다 LSTM이 기억 문제에 더 유리한 경우가 많습니다.

다음 단계로는 이 코드에 망각 게이트, 입력 게이트, 출력 게이트 수식을 연결해서 보면 좋습니다.